In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_gold.region_risk_scores AS
WITH cfg AS (
  SELECT
    MAX(CASE WHEN param_name='seismic_mag_moderate' THEN param_value END) AS s_mod,
    MAX(CASE WHEN param_name='seismic_mag_severe'   THEN param_value END) AS s_sev,
    MAX(CASE WHEN param_name='wind_moderate_kmh'    THEN param_value END) AS w_mod,
    MAX(CASE WHEN param_name='wind_severe_kmh'      THEN param_value END) AS w_sev,
    MAX(CASE WHEN param_name='precip_moderate_mm'   THEN param_value END) AS p_mod,
    MAX(CASE WHEN param_name='precip_severe_mm'     THEN param_value END) AS p_sev,
    MAX(CASE WHEN param_name='weight_seismic'       THEN param_value END) AS w_seismic,
    MAX(CASE WHEN param_name='weight_wind'          THEN param_value END) AS w_wind,
    MAX(CASE WHEN param_name='weight_precip'        THEN param_value END) AS w_precip,
    MAX(CASE WHEN param_name='weight_base'          THEN param_value END) AS w_base,
    MAX(CASE WHEN param_name='band_elevated_min'    THEN param_value END) AS b_elev,
    MAX(CASE WHEN param_name='band_high_min'        THEN param_value END) AS b_high
  FROM workspace.petropulse_ref.risk_thresholds
),
seis AS (
  SELECT region_id, COUNT(*) AS nearby_events, MAX(magnitude) AS max_magnitude
  FROM workspace.petropulse_silver.region_seismic_nearby
  GROUP BY region_id
),
base AS (
  SELECT
    r.region_id, r.region_name, r.base_risk_indicator,
    COALESCE(s.nearby_events, 0) AS nearby_events,
    COALESCE(s.max_magnitude, 0) AS max_magnitude,
    COALESCE(w.wind_speed_kmh, 0) AS wind_speed_kmh,
    COALESCE(w.precipitation_mm, 0) AS precipitation_mm,
    c.*
  FROM workspace.petropulse_ref.operating_regions r
  LEFT JOIN seis s ON s.region_id = r.region_id
  LEFT JOIN workspace.petropulse_silver.weather_latest w ON w.region_id = r.region_id
  CROSS JOIN cfg c
),
comp AS (
  SELECT *,
    CASE WHEN max_magnitude <= 0 THEN 0
         WHEN max_magnitude < s_mod THEN 50 * max_magnitude / s_mod
         WHEN max_magnitude < s_sev THEN 50 + 50 * (max_magnitude - s_mod) / (s_sev - s_mod)
         ELSE 100 END AS seismic_score,
    CASE WHEN wind_speed_kmh < w_mod THEN 50 * wind_speed_kmh / w_mod
         WHEN wind_speed_kmh < w_sev THEN 50 + 50 * (wind_speed_kmh - w_mod) / (w_sev - w_mod)
         ELSE 100 END AS wind_score,
    CASE WHEN precipitation_mm < p_mod THEN 50 * precipitation_mm / p_mod
         WHEN precipitation_mm < p_sev THEN 50 + 50 * (precipitation_mm - p_mod) / (p_sev - p_mod)
         ELSE 100 END AS precip_score
  FROM base
)
SELECT
  region_id, region_name, nearby_events, max_magnitude, wind_speed_kmh, precipitation_mm,
  ROUND(seismic_score, 1) AS seismic_score,
  ROUND(wind_score, 1) AS wind_score,
  ROUND(precip_score, 1) AS precip_score,
  base_risk_indicator,
  ROUND(w_seismic*seismic_score + w_wind*wind_score + w_precip*precip_score + w_base*base_risk_indicator, 1) AS risk_score,
  CASE WHEN w_seismic*seismic_score + w_wind*wind_score + w_precip*precip_score + w_base*base_risk_indicator >= b_high THEN 'High'
       WHEN w_seismic*seismic_score + w_wind*wind_score + w_precip*precip_score + w_base*base_risk_indicator >= b_elev THEN 'Elevated'
       ELSE 'Low' END AS risk_band,
  current_timestamp() AS scored_at
FROM comp;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_gold.price_volatility AS
WITH cfg AS (
  SELECT
    MAX(CASE WHEN param_name='price_change_moderate_pct' THEN param_value END) AS mod_pct,
    MAX(CASE WHEN param_name='price_change_high_pct'     THEN param_value END) AS high_pct
  FROM workspace.petropulse_ref.risk_thresholds
)
SELECT
  p.price_date, p.benchmark, p.price_usd_bbl, p.pct_change,
  CASE WHEN p.pct_change IS NULL THEN 'Unknown'
       WHEN ABS(p.pct_change) >= c.high_pct THEN 'High'
       WHEN ABS(p.pct_change) >= c.mod_pct  THEN 'Moderate'
       ELSE 'Low' END AS volatility_impact
FROM workspace.petropulse_silver.crude_prices p
CROSS JOIN cfg c;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_gold.high_risk_regions AS
SELECT * FROM workspace.petropulse_gold.region_risk_scores
WHERE risk_band IN ('High', 'Elevated');

In [0]:
%sql
SELECT region_id, region_name, nearby_events, max_magnitude, wind_speed_kmh, precipitation_mm,
       seismic_score, wind_score, precip_score, base_risk_indicator, risk_score, risk_band
FROM workspace.petropulse_gold.region_risk_scores
ORDER BY risk_score DESC;

In [0]:
%sql
SELECT price_date, benchmark, price_usd_bbl, pct_change, volatility_impact
FROM workspace.petropulse_gold.price_volatility
ORDER BY price_date DESC, benchmark
LIMIT 10;